<a href="https://colab.research.google.com/github/GanghyunShin02/Thermodynamic_Realgas/blob/main/Thermodynamics_assignment.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import sympy as sp
import numpy as np


B,C,P,T=140*1e-6,7200*1e-12,12e+5,298
R=8.314

P_sym,V_sym,T_sym,R_sym,Z_sym,B_sym,C_sym=sp.symbols('P,V,T,R,Z,B,C')
Bp_sym,Cp_sym=sp.symbols("B',C'")

Z1=1+B_sym/V_sym +C_sym/V_sym**2
Z2=1+Bp_sym*P_sym+Cp_sym*P_sym**2

Z1_func=sp.lambdify([B_sym,C_sym,V_sym],Z1,'numpy')
Z2_func=sp.lambdify([P_sym,Bp_sym,Cp_sym],Z2,'numpy')


#Z1에서 P를 V에 관한 식으로 정리
a=sp.solve(Z1-(P_sym*V_sym/(R_sym*T_sym)),P_sym)
print('Z1에서 P를 V에 관한 식으로 정리')
print(sp.expand(a[0]))

#Z2에 정리한 P를 대입

aa=Z2.subs(P_sym,a[0])
print('Z2에 정리한 P를 대입')
print(sp.collect(aa,1/V_sym))

#Z2-Z1을 전개..

aaa=sp.collect(aa-Z1,Bp_sym)
print('Z2-Z1을 전개..')

print(sp.series(aaa,1/V_sym))

## 항별비교로 프라임계수를 알아냄.

Bpp_func=sp.lambdify((B_sym,R_sym,T_sym),B_sym/(R_sym*T_sym),'numpy')
Cpp_func=sp.lambdify((C_sym,R_sym,T_sym,B_sym),(C_sym-B_sym**2)/(R_sym*T_sym)**2,'numpy')

zz=1+Bpp_func(B,R,T)*P+Cpp_func(C,R,T,B)*P**2
print('압축인자',zz)

v=sp.solve(zz-Z1,V_sym)
print('부피식',v)
v_func=sp.lambdify((B_sym,C_sym,R_sym,T_sym),v,'numpy')
print('부피',v_func(B,C,R,T))

Z1에서 P를 V에 관한 식으로 정리
B*R*T/V**2 + C*R*T/V**3 + R*T/V
Z2에 정리한 P를 대입
B'*R*T*(B*V + C + V**2)/V**3 + C'*R**2*T**2*(B*V + C + V**2)**2/V**6 + 1
Z2-Z1을 전개..
(2*B*C'*R**2*T**2 + B'*C*R*T)/V**3 + (B*B'*R*T - C + C'*R**2*T**2)/V**2 + (-B + B'*R*T)/V + C'*R**2*T**2*(B**2 + 2*C)/V**4 + 2*B*C*C'*R**2*T**2/V**5 + O(V**(-6), (V, oo))
압축인자 1.0648994031494337
부피식 [7.70423109822333*B - 1.54084621964467e-7*sqrt(2.5e+15*B**2 + 648994031494337.0*C), 7.70423109822333*B + 1.54084621964467e-7*sqrt(2.5e+15*B**2 + 648994031494337.0*C)]
부피 [np.float64(-5.0257677645783186e-05), np.float64(0.0022074423851483156)]


In [ ]:
#pitzer

z_pitzer=1+(B_sym*P_sym)/(R_sym*T_sym)
print('pitzer식에서 z는',z_pitzer)
z_pitzer_f=sp.lambdify((B_sym,P_sym,R_sym,T_sym),z_pitzer,'numpy')
print('그 값은',z_pitzer_f(B,P,8.314,T))

pitzer식에서 z는 B*P/(R*T) + 1
그 값은 1.0678083220184922


In [ ]:
#RK

b_sym,PSI_sym,OMEGA_sym,Tc_sym,Tr_sym,Pc_sym=sp.symbols('b,PSI,OMEGA,Tc,Tr,Pc')

Tc,Pc=282.3,50.4e+5
Tr,Pr=T/Tc,P/Pc

PSI=0.42748
OMEGA=0.08664


#P를 변수들로 나타냄..
alp=Tr_sym**-0.5


aT=PSI_sym*(alp*R_sym**2*Tc_sym**2/(Pc_sym))

b_RK=OMEGA_sym*R_sym*Tc_sym/Pc_sym
P_RK=(R_sym*T_sym/(V_sym-b_RK))-aT/(V_sym*(V_sym+b_RK))
print('RK에서 압력',P_RK)
P_RK_f=sp.lambdify((V_sym,PSI_sym,OMEGA_sym,T_sym,Tc_sym,Tr_sym,Pc_sym,R_sym),P_RK,'numpy')

#V에관하여 정리

try:
  expr_Vn=sp.nsolve(P_RK_f(V_sym,PSI,OMEGA,T,Tc,Tr,Pc,R)-P,V_sym,0.0022074423851483156)
except:RuntimeError

expr_V=sp.solve(P_RK-P,V_sym)

print('부피에관하여정리하면',expr_V)
print('t수치해',expr_Vn)

expr_V_f=sp.lambdify((PSI_sym,OMEGA_sym,T_sym,Tc_sym,Tr_sym,Pc_sym,R_sym),expr_V,'numpy')



VV=expr_V_f(PSI,OMEGA,T,Tc,Tr,Pc,R)
print(VV)

V_RK=VV[1].real
z_RK=P*V_RK/(R*T)
print('z는',z_RK)


RK에서 압력 -PSI*R**2*Tc**2/(Pc*Tr**0.5*V*(OMEGA*R*Tc/Pc + V)) + R*T/(-OMEGA*R*Tc/Pc + V)
부피에관하여정리하면 [2.77777777777778e-7*R*T - 0.333333333333333*(6.94444444444444e-13*R**2*T**2 - 2.5e-6*(-1200000.0*OMEGA**2*R**2*Tc**2*sqrt(Tr) - OMEGA*Pc*R**2*T*Tc*sqrt(Tr) + PSI*Pc*R**2*Tc**2)/(Pc**2*sqrt(Tr)))/(-1.125e-5*OMEGA*PSI*R**3*Tc**3/(Pc**2*Tr**0.5) - 5.78703703703704e-19*R**3*T**3 + 0.5*(-4.0*(6.94444444444444e-13*R**2*T**2 - 2.5e-6*(-1200000.0*OMEGA**2*R**2*Tc**2*Tr**0.5 - OMEGA*Pc*R**2*T*Tc*Tr**0.5 + PSI*Pc*R**2*Tc**2)/(Pc**2*Tr**0.5))**3 + (-2.25e-5*OMEGA*PSI*R**3*Tc**3/(Pc**2*Tr**0.5) - 1.15740740740741e-18*R**3*T**3 + 6.25e-12*R*T*(-1200000.0*OMEGA**2*R**2*Tc**2*Tr**0.5 - OMEGA*Pc*R**2*T*Tc*Tr**0.5 + PSI*Pc*R**2*Tc**2)/(Pc**2*Tr**0.5))**2)**0.5 + 3.125e-12*R*T*(-1200000.0*OMEGA**2*R**2*Tc**2*Tr**0.5 - OMEGA*Pc*R**2*T*Tc*Tr**0.5 + PSI*Pc*R**2*Tc**2)/(Pc**2*Tr**0.5))**(1/3) - 0.333333333333333*(-1.125e-5*OMEGA*PSI*R**3*Tc**3/(Pc**2*Tr**0.5) - 5.78703703703704e-19*R**3*T**3 + 0.5*(-4.0*(6.9444

NameError: name 'expr_Vn' is not defined

In [ ]:
#SRK
#수치해로 하겠음..

omega_sym=sp.symbols('omega')
omega=0.087


alp=(1+(0.48+1.574*omega_sym-0.176*omega_sym**2)*(1-Tr**(0.5)))**2
aT_SRK=PSI_sym*(alp*R_sym**2*Tc_sym**2/(Pc_sym))
b_SRK=OMEGA_sym*R_sym*Tc_sym/Pc_sym
P_SRK=(R_sym*T_sym/(V_sym-b_SRK))-aT_SRK/(V_sym*(V_sym+b_SRK))

P_SRK_f=sp.lambdify((V_sym,PSI_sym,OMEGA_sym,T_sym,Tc_sym,Tr_sym,Pc_sym,R_sym,omega_sym),P_SRK,'numpy')

try:
  V_SRKn=sp.nsolve(P_SRK_f(V_sym,PSI,OMEGA,T,Tc,Tr,Pc,R,omega)-P,V_sym,0.0022074423851483156)
except:RuntimeError

V_SRK=sp.solve(P_SRK-P,V_sym)
print('부피에관하여정리하면',V_SRK)

V_SRK_f=sp.lambdify((PSI_sym,OMEGA_sym,T_sym,Tc_sym,Tr_sym,Pc_sym,R_sym,omega_sym),V_SRK,'numpy')

Vsrk=V_SRK_f(PSI,OMEGA,T,Tc,Tr,Pc,R,omega)

print("부피",Vsrk)
print('t수치해',V_SRKn)
Z_SRK=P*Vsrk[1].real/(R*T)
print('압축인자',Z_SRK)


부피에관하여정리하면 [2.77777777777778e-7*R*T - 0.333333333333333*(6.94444444444444e-13*R**2*T**2 - 1.0e-39*(-3.0e+39*OMEGA**2*R**2*Tc**2 - 2.5e+33*OMEGA*Pc*R**2*T*Tc + 5.82707622703821e+28*PSI*Pc*R**2*Tc**2*omega**4 - 1.04225204333615e+30*PSI*Pc*R**2*Tc**2*omega**3 + 2.84820219013239e+31*PSI*Pc*R**2*Tc**2*omega**2 - 2.13039980098599e+32*PSI*Pc*R**2*Tc**2*omega + 2.4345988616361e+33*PSI*Pc*R**2*Tc**2)/Pc**2)/(-5.78703703703704e-19*R**3*T**3 + 0.5*(-4.0*(6.94444444444444e-13*R**2*T**2 - 1.0e-39*(-3.0e+39*OMEGA**2*R**2*Tc**2 - 2.5e+33*OMEGA*Pc*R**2*T*Tc + 5.82707622703821e+28*PSI*Pc*R**2*Tc**2*omega**4 - 1.04225204333615e+30*PSI*Pc*R**2*Tc**2*omega**3 + 2.84820219013239e+31*PSI*Pc*R**2*Tc**2*omega**2 - 2.13039980098599e+32*PSI*Pc*R**2*Tc**2*omega + 2.4345988616361e+33*PSI*Pc*R**2*Tc**2)/Pc**2)**3 + (-1.15740740740741e-18*R**3*T**3 + 2.5e-45*R*T*(-3.0e+39*OMEGA**2*R**2*Tc**2 - 2.5e+33*OMEGA*Pc*R**2*T*Tc + 5.82707622703821e+28*PSI*Pc*R**2*Tc**2*omega**4 - 1.04225204333615e+30*PSI*Pc*R**2*Tc**2*omega

In [ ]:
#PR
sigma=1-2**0.5
epsilon=1-2**0.5
OMEGA_PR=0.07780
PSI_PR=0.45724

alp=(1+(0.3764+1.54426*omega_sym-0.26992*omega_sym**2)*(1-Tr_sym**(0.5)))**2
aT_PR=PSI_sym*(alp*R_sym**2*Tc_sym**2/(Pc_sym))
b_PR=OMEGA_sym*R_sym*Tc_sym/Pc_sym
P_PR=(R_sym*T_sym/(V_sym-b_PR))-aT_PR/((V_sym+epsilon*b_PR)*(V_sym+sigma*b_PR))

P_PR_f=sp.lambdify((V_sym,PSI_sym,OMEGA_sym,T_sym,Tc_sym,Tr_sym,Pc_sym,R_sym,omega_sym),P_PR,'numpy')

try:

  V_PRn=sp.nsolve(P_PR_f(V_sym,PSI_PR,OMEGA_PR,T,Tc,Tr,Pc,R,omega)-P,V_sym,0.0022074423851483156)
except: RuntimeError

print('t수치해',V_PRn)

V_PR=sp.solve(P_PR-P,V_sym)
print('g해석해',V_PR)

V_PR_f=sp.lambdify((PSI_sym,OMEGA_sym,T_sym,Tc_sym,Tr_sym,Pc_sym,R_sym,omega_sym),V_PR,'numpy')

V_PRR=V_PR_f(PSI_PR,OMEGA_PR,T,Tc,Tr,Pc,R,omega)
print('g해석해',V_PRR)


Z_PR=P*V_PRR[1].real/(R*T)
print('압축인자',Z_PR)




t수치해 188.702480954023
g해석해 [-0.333333333333333*(1.11111111111111e-29*(-548528137423857.0*OMEGA*R*Tc - 250000000.0*Pc*R*T)**2/Pc**2 - 2.5e-29*(1.2e+29*OMEGA**2*R**2*Tc**2 + 8.2842712474619e+22*OMEGA*Pc*R**2*T*Tc - 1.457136128e+22*PSI*Pc*R**2*Tc**2*sqrt(Tr)*omega**4 + 1.6673066368e+23*PSI*Pc*R**2*Tc**2*sqrt(Tr)*omega**3 - 3.8232463432e+23*PSI*Pc*R**2*Tc**2*sqrt(Tr)*omega**2 - 5.413557856e+23*PSI*Pc*R**2*Tc**2*sqrt(Tr)*omega - 1.03615392e+23*PSI*Pc*R**2*Tc**2*sqrt(Tr) + 7.28568064e+21*PSI*Pc*R**2*Tc**2*Tr*omega**4 - 8.336533184e+22*PSI*Pc*R**2*Tc**2*Tr*omega**3 + 2.1815431716e+23*PSI*Pc*R**2*Tc**2*Tr*omega**2 + 1.162518928e+23*PSI*Pc*R**2*Tc**2*Tr*omega + 1.4167696e+22*PSI*Pc*R**2*Tc**2*Tr + 7.28568064e+21*PSI*Pc*R**2*Tc**2*omega**4 - 8.336533184e+22*PSI*Pc*R**2*Tc**2*omega**3 + 1.6417031716e+23*PSI*Pc*R**2*Tc**2*omega**2 + 4.251038928e+23*PSI*Pc*R**2*Tc**2*omega + 1.89447696e+23*PSI*Pc*R**2*Tc**2)/Pc**2)/(0.5*(-4.0*(1.11111111111111e-29*(-548528137423857.0*OMEGA*R*Tc - 250000000.0*Pc*R*T

In [ ]:
#Ethane_RK

P_E,V_E,T_E=140e+5,0.15,333
Pc_E,Vc_E,Tc_E=48.72e+5,145.5*1e-6,305.3
Pr_E,Tr_E=P_E/Pc_E, T_E/Tc_E
M_E=30.070

omega_E=0.1

#P를 변수들로 나타냄..
alp=Tr_sym**-0.5


aT=PSI_sym*(alp*R_sym**2*Tc_sym**2/(Pc_sym))

b_RK=OMEGA_sym*R_sym*Tc_sym/Pc_sym
P_RK=(R_sym*T_sym/(V_sym-b_RK))-aT/(V_sym*(V_sym+b_RK))
print('RK에서 압력',P_RK)
P_RK_f=sp.lambdify((V_sym,PSI_sym,OMEGA_sym,T_sym,Tc_sym,Tr_sym,Pc_sym,R_sym),P_RK,'numpy')

#V에관하여 정리

try:
  expr_Vn=sp.nsolve(P_RK_f(V_sym,PSI,OMEGA,T_E,Tc_E,Tr_E,Pc_E,R)-P_E,V_sym,0.00017)
except:RuntimeError

expr_V=sp.solve(P_RK-P_E,V_sym)

print('부피에관하여정리하면',expr_V)
print('t수치해',expr_Vn)

expr_V_f=sp.lambdify((PSI_sym,OMEGA_sym,T_sym,Tc_sym,Tr_sym,Pc_sym,R_sym),expr_V,'numpy')


VV=expr_V_f(PSI,OMEGA,T_E,Tc_E,Tr_E,Pc_E,R)
print(VV)

V_RK=VV[0].real
z_RK=P*V_RK/(R*T)
print('z는',z_RK)

print('몰부피가 위와같으므로 에탄은',V_E/VV[0].real,'mol 있다.')
print('분자량이 30.070g/mol이므로',M_E*(V_E/VV[0].real),'g 존재한다.')



RK에서 압력 -PSI*R**2*Tc**2/(Pc*Tr**0.5*V*(OMEGA*R*Tc/Pc + V)) + R*T/(-OMEGA*R*Tc/Pc + V)
부피에관하여정리하면 [2.38095238095238e-8*R*T - 0.333333333333333*(5.10204081632653e-15*R**2*T**2 - 2.14285714285714e-7*(-14000000.0*OMEGA**2*R**2*Tc**2*sqrt(Tr) - OMEGA*Pc*R**2*T*Tc*sqrt(Tr) + PSI*Pc*R**2*Tc**2)/(Pc**2*sqrt(Tr)))/(-9.64285714285714e-7*OMEGA*PSI*R**3*Tc**3/(Pc**2*Tr**0.5) - 3.64431486880466e-22*R**3*T**3 + 0.5*(-4.0*(5.10204081632653e-15*R**2*T**2 - 2.14285714285714e-7*(-14000000.0*OMEGA**2*R**2*Tc**2*Tr**0.5 - OMEGA*Pc*R**2*T*Tc*Tr**0.5 + PSI*Pc*R**2*Tc**2)/(Pc**2*Tr**0.5))**3 + (-1.92857142857143e-6*OMEGA*PSI*R**3*Tc**3/(Pc**2*Tr**0.5) - 7.28862973760933e-22*R**3*T**3 + 4.59183673469388e-14*R*T*(-14000000.0*OMEGA**2*R**2*Tc**2*Tr**0.5 - OMEGA*Pc*R**2*T*Tc*Tr**0.5 + PSI*Pc*R**2*Tc**2)/(Pc**2*Tr**0.5))**2)**0.5 + 2.29591836734694e-14*R*T*(-14000000.0*OMEGA**2*R**2*Tc**2*Tr**0.5 - OMEGA*Pc*R**2*T*Tc*Tr**0.5 + PSI*Pc*R**2*Tc**2)/(Pc**2*Tr**0.5))**(1/3) - 0.333333333333333*(-9.64285714285714e-7*OM

NameError: name 'expr_Vn' is not defined

In [ ]:
#SRK_Ethane

alp=(1+(0.48+1.574*omega_sym-0.176*omega_sym**2)*(1-Tr**(0.5)))**2
aT_SRK=PSI_sym*(alp*R_sym**2*Tc_sym**2/(Pc_sym))
b_SRK=OMEGA_sym*R_sym*Tc_sym/Pc_sym
P_SRK=(R_sym*T_sym/(V_sym-b_SRK))-aT_SRK/(V_sym*(V_sym+b_SRK))

P_SRK_f=sp.lambdify((V_sym,PSI_sym,OMEGA_sym,T_sym,Tc_sym,Tr_sym,Pc_sym,R_sym,omega_sym),P_SRK,'numpy')

try:
  V_SRKn=sp.nsolve(P_SRK_f(V_sym,PSI,OMEGA,T_E,Tc_E,Tr_E,Pc_E,R,omega_E)-P_E,V_sym,0.0022074423851483156)
except:RuntimeError

V_SRK=sp.solve(P_SRK-P_E,V_sym)
print('부피에관하여정리하면',V_SRK)

V_SRK_f=sp.lambdify((PSI_sym,OMEGA_sym,T_sym,Tc_sym,Tr_sym,Pc_sym,R_sym,omega_sym),V_SRK,'numpy')

Vsrk=V_SRK_f(PSI,OMEGA,T_E,Tc_E,Tr_E,Pc_E,R,omega_E)

print("부피",Vsrk)
print('t수치해',V_SRKn)
Z_SRK=P*Vsrk[0].real/(R*T)
print('압축인자',Z_SRK)

print('몰부피가 위와같으므로 에탄은',V_E/Vsrk[0].real,'mol 있다.')
print('분자량이 30.070g/mol이므로',M_E*(V_E/Vsrk[0].real),'g 존재한다.')




부피에관하여정리하면 [2.38095238095238e-8*R*T - 0.333333333333333*(5.10204081632653e-15*R**2*T**2 - 8.57142857142857e-41*(-3.5e+40*OMEGA**2*R**2*Tc**2 - 2.5e+33*OMEGA*Pc*R**2*T*Tc + 5.82707622703821e+28*PSI*Pc*R**2*Tc**2*omega**4 - 1.04225204333615e+30*PSI*Pc*R**2*Tc**2*omega**3 + 2.84820219013239e+31*PSI*Pc*R**2*Tc**2*omega**2 - 2.13039980098599e+32*PSI*Pc*R**2*Tc**2*omega + 2.4345988616361e+33*PSI*Pc*R**2*Tc**2)/Pc**2)/(-3.64431486880466e-22*R**3*T**3 + 0.5*(-4.0*(5.10204081632653e-15*R**2*T**2 - 8.57142857142857e-41*(-3.5e+40*OMEGA**2*R**2*Tc**2 - 2.5e+33*OMEGA*Pc*R**2*T*Tc + 5.82707622703821e+28*PSI*Pc*R**2*Tc**2*omega**4 - 1.04225204333615e+30*PSI*Pc*R**2*Tc**2*omega**3 + 2.84820219013239e+31*PSI*Pc*R**2*Tc**2*omega**2 - 2.13039980098599e+32*PSI*Pc*R**2*Tc**2*omega + 2.4345988616361e+33*PSI*Pc*R**2*Tc**2)/Pc**2)**3 + (-7.28862973760933e-22*R**3*T**3 + 1.83673469387755e-47*R*T*(-3.5e+40*OMEGA**2*R**2*Tc**2 - 2.5e+33*OMEGA*Pc*R**2*T*Tc + 5.82707622703821e+28*PSI*Pc*R**2*Tc**2*omega**4 - 1.042

In [ ]:
#PR_Ethane

alp=(1+(0.3764+1.54426*omega_sym-0.26992*omega_sym**2)*(1-Tr_sym**(0.5)))**2
aT_PR=PSI_sym*(alp*R_sym**2*Tc_sym**2/(Pc_sym))
b_PR=OMEGA_sym*R_sym*Tc_sym/Pc_sym
P_PR=(R_sym*T_sym/(V_sym-b_PR))-aT_PR/((V_sym+epsilon*b_PR)*(V_sym+sigma*b_PR))

P_PR_f=sp.lambdify((V_sym,PSI_sym,OMEGA_sym,T_sym,Tc_sym,Tr_sym,Pc_sym,R_sym,omega_sym),P_PR,'numpy')

try:
  V_PRn=sp.nsolve(P_PR_f(V_sym,PSI_PR,OMEGA_PR,T_E,Tc_E,Tr_E,Pc_E,R,omega_E)-P_E,V_sym,0.00017993871428571427)
except: RuntimeError

print('t수치해',V_PRn)

V_PR=sp.solve(P_PR-P_E,V_sym)
args=[PSI_sym,OMEGA_sym,T_sym,Tc_sym,Tr_sym,Pc_sym,R_sym,omega_sym,V_PR]
V_PR_f=sp.lambdify((PSI_sym,OMEGA_sym,T_sym,Tc_sym,Tr_sym,Pc_sym,R_sym,omega_sym),V_PR,'numpy')

Va=V_PR_f(PSI_PR,OMEGA_PR,T_E,Tc_E,Tr_E,Pc_E,R,omega_E)
print('g해석해',Va)

Z_PR=P*Va[0].real/(R*T)
print('압축인자',Z_PR)

print('몰부피가 위와같으므로 에탄은',V_E/Va[0].real,'mol 있다.')
print('분자량이 30.070g/mol이므로',M_E*(V_E/Va[1].real),'g 존재한다.')




t수치해 188.702480954023
g해석해 [np.float64(4.403360220096084e-05), (0.0001139162650525839-0.00016267484025758697j), (0.0001139162650525839+0.00016267484025758697j)]
압축인자 0.02132746198340674
몰부피가 위와같으므로 에탄은 3406.4894194989774 mol 있다.
분자량이 30.070g/mol이므로 39594.87258398041 g 존재한다.


In [ ]:
#b번 RK


#에탄이 40kg있고 분자량이 30.70이므로 몰부피를구한다.
P2,V2=200e5,0.15/(40000/M_E)
Tr2=T_sym/Tc_E

alp=Tr2**-0.5

aT=PSI_sym*(alp*R_sym**2*Tc_sym**2/(Pc_sym))

b_RK=OMEGA_sym*R_sym*Tc_sym/Pc_sym
P_RK=(R_sym*T_sym/(V_sym-b_RK))-aT/(V_sym*(V_sym+b_RK))
print('RK에서 압력',P_RK)
P_RK_f=sp.lambdify((V_sym,PSI_sym,OMEGA_sym,T_sym,Tc_sym,Pc_sym,R_sym),P_RK,'numpy')

#T에관하여 정리

try:
  expr_Tn=sp.nsolve(P_RK_f(V2,PSI,OMEGA,T_sym,Tc_E,Pc_E,R)-P2,T_sym,271)
except:RuntimeError

'''
expr_T=sp.solve(P_RK-P2,T_sym)

print('온도에관하여정리하면',expr_T)
'''
print('t수치해',expr_Tn)
print('dkq',P2*V2/(R*expr_Tn))

#expr_T_f=sp.lambdify((PSI_sym,OMEGA_sym,V_sym,Tc_sym,Pc_sym,R_sym),expr_T,'numpy')

#TT=expr_T_f(PSI,OMEGA,V2,Tc_E,Pc_E,R)
#print(expr_Tn)
'''
T_RK=TT[1].real
print('구하는온도는',T_RK)
z_RK=P*V2/(R*T_RK)
print('z는',z_RK)
'''

RK에서 압력 -17.4728360605827*PSI*R**2*Tc**2/(Pc*T**0.5*V*(OMEGA*R*Tc/Pc + V)) + R*T/(-OMEGA*R*Tc/Pc + V)
t수치해 390.895751467793
dkq 0.693942875069908


"\nT_RK=TT[1].real\nprint('구하는온도는',T_RK)\nz_RK=P*V2/(R*T_RK)\nprint('z는',z_RK)\n"

In [ ]:
#b번 SRK

omega_sym=sp.symbols('omega')


alp=(1+(0.48+1.574*omega_sym-0.176*omega_sym**2)*(1-Tr2**(0.5)))**2
aT_SRK=PSI_sym*(alp*R_sym**2*Tc_sym**2/(Pc_sym))
b_SRK=OMEGA_sym*R_sym*Tc_sym/Pc_sym
P_SRK=(R_sym*T_sym/(V_sym-b_SRK))-aT_SRK/(V_sym*(V_sym+b_SRK))

P_SRK_f=sp.lambdify((V_sym,PSI_sym,OMEGA_sym,T_sym,Tc_sym,Pc_sym,R_sym,omega_sym),P_SRK,'numpy')

try:
  V_SRKn=sp.nsolve(P_SRK_f(V2,PSI,OMEGA,T_sym,Tc_E,Pc_E,R,omega_E)-P2,T_sym,271)
except:RuntimeError

print(V_SRKn)
print('dkq',P2*V2/(R*V_SRKn))
#print(V_SRKn(V2,PSI,O))

'''
T_SRK=sp.solve(P_SRK-P2,T_sym)
print('온도에관하여정리하면',T_SRK)

T_SRK_f=sp.lambdify((PSI_sym,OMEGA_sym,V_sym,Tc_sym,Pc_sym,R_sym,omega_sym),T_SRK,'numpy')

Tsrk=T_SRK_f(PSI,OMEGA,V2,Tc_E,Pc_E,R,omega_E)
'''
''
'''
print("온도",Tsrk)
print('t수치해',V_SRKn)
Z_SRK=P*Tsrk[1].real/(R*T)
print('압축인자',Z_SRK)

print('구하는 온도는',Tsrk[1].real)
'''

383.005070315747
dkq 0.708239505556802


'\nprint("온도",Tsrk)\nprint(\'t수치해\',V_SRKn)\nZ_SRK=P*Tsrk[1].real/(R*T)\nprint(\'압축인자\',Z_SRK)\n\nprint(\'구하는 온도는\',Tsrk[1].real)\n'

In [ ]:
#b번 PR
sigma=1-2**0.5
epsilon=1-2**0.5
OMEGA_PR=0.07780
PSI_PR=0.45724


alp=(1+(0.3764+1.54426*omega_sym-0.26992*omega_sym**2)*(1-Tr2**(0.5)))**2
aT_PR=PSI_sym*(alp*R_sym**2*Tc_sym**2/(Pc_sym))
b_PR=OMEGA_sym*R_sym*Tc_sym/Pc_sym
P_PR=(R_sym*T_sym/(V_sym-b_PR))-aT_PR/((V_sym+epsilon*b_PR)*(V_sym+sigma*b_PR))

P_PR_f=sp.lambdify((V_sym,PSI_sym,OMEGA_sym,T_sym,Tc_sym,Pc_sym,R_sym,omega_sym),P_PR,'numpy')

try:
  V_PRn=sp.nsolve(P_PR_f(V2,PSI,OMEGA,T_sym,Tc_E,Pc_E,R,omega_E)-P_E,T_sym,271)
except: RuntimeError

print('t수치해',V_PRn)
print('dkq',P2*V2/(R*V_PRn))

'''
T_PR=sp.solve(P_PR-P2,T_sym)

T_PR_f=sp.lambdify((PSI_sym,OMEGA_sym,Tc_sym,V_sym,Pc_sym,R_sym,omega_sym),T_PR,'numpy')

Ta=T_PR_f(PSI_PR,OMEGA_PR,Tc_E,V2,Pc_E,R,omega_E)
print('g해석해',Ta)

print('구하는온도는',Ta[1].real)
Z_PR=P*V2/(R*Ta[1].real)
print('압축인자',Z_PR)

'''



t수치해 494.581101617593
dkq 0.548462771300770


"\nT_PR=sp.solve(P_PR-P2,T_sym)\n\nT_PR_f=sp.lambdify((PSI_sym,OMEGA_sym,Tc_sym,V_sym,Pc_sym,R_sym,omega_sym),T_PR,'numpy')\n\nTa=T_PR_f(PSI_PR,OMEGA_PR,Tc_E,V2,Pc_E,R,omega_E)\nprint('g해석해',Ta)\n\nprint('구하는온도는',Ta[1].real)\nZ_PR=P*V2/(R*Ta[1].real)\nprint('압축인자',Z_PR)\n\n"